# Phase 2b — Urban Context Analysis

**BACKEND_PLAN.md §2b** · The site does not exist in isolation — it sits inside a city.

This notebook validates the full urban context pipeline introduced in Phase 2b:

1. **Road network** — roads coloured by hierarchy (main / secondary / path)
2. **Intersection classification** — crossroads, T-junctions, Y-junctions from geometry
3. **Frontage analysis** — which site sides face the city, scored by visibility
4. **Corner conditions** — gateway corners where a main road is involved
5. **Access recommendations** — vehicle / pedestrian / service entry points
6. **Urban response** — architectural massing, entry, and facade strategies per site type

Three synthetic sites demonstrate distinct urban conditions:
- **Site A** — Crossroads corner (Eixample-style, Barcelona): main + secondary road meeting
- **Site B** — T-junction terminal (Bloomsbury-style, London): site addresses the visual axis of a terminated road
- **Site C** — Triangular corner (Flatiron-style, New York): two main roads meeting at an acute angle

No LLM, no MCP. All geometry is deterministic.  
OSM fetch (optional) is attempted for Barcelona; falls back to synthetic if offline.

In [1]:
import sys, os
from pathlib import Path

# Locate workspace root (directory containing team_04/).
# Works whether Jupyter cwd is AIA26_Studio, team_04, or team_04/test_notebooks.
_cwd = Path(os.getcwd()).resolve()
_root = next(
    (p for p in [_cwd, _cwd.parent, _cwd.parent.parent] if (p / 'team_04').is_dir()),
    None,
)
if _root is None:
    raise FileNotFoundError(
        'Cannot locate workspace root (directory containing team_04/). '
        f'Tried: {_cwd}, {_cwd.parent}, {_cwd.parent.parent}'
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

print(f'Workspace root: {_root}')

Workspace root: C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

from team_04.agent.tools.site_model import build_site_model
from team_04.agent.tools.road_context import analyze_roads
from team_04.agent.tools.urban_analysis import (
    full_urban_analysis,
    detect_intersections_from_roads,
)
from team_04.agent.tools.osm_context import (
    SYNTHETIC_SITES,
    INTERESTING_SITES,
    fetch_or_fallback,
)

# ── Colour palettes ────────────────────────────────────────────────────────
HIER_COLOUR = {'main': '#E63946', 'secondary': '#F4A261', 'path': '#2A9D8F'}
IX_COLOUR   = {
    'crossroads':       '#9B2226',
    't_junction':       '#AE2012',
    'y_junction':       '#CA6702',
    'complex_junction': '#6A0572',
    'dead_end':         '#555555',
    'bend':             '#AAAAAA',
}
IX_MARKER   = {'crossroads': 'X', 't_junction': 'v', 'y_junction': 'Y',
               'complex_junction': '*', 'dead_end': 'o', 'bend': 's'}
ACCESS_COLOUR = {'vehicle': '#264653', 'pedestrian': '#2A9D8F', 'service': '#E9C46A'}
VIS_CMAP = LinearSegmentedColormap.from_list('vis', [
    (244/255, 226/255, 133/255),   # vis=0 → yellow
    (42/255,  157/255, 143/255),   # vis=1 → teal
])


def _vis_colour(score: float) -> tuple:
    """Visibility score 0..1 → yellow-to-teal RGB tuple."""
    r = (244 + (42  - 244) * score) / 255
    g = (226 + (157 - 226) * score) / 255
    b = (133 + (143 - 133) * score) / 255
    return (r, g, b)


def _wrap(text: str, width: int = 60) -> str:
    """Basic word-wrap for matplotlib text blocks."""
    words = text.split()
    lines_out, current = [], []
    for w in words:
        if sum(len(x) + 1 for x in current) + len(w) > width:
            lines_out.append(' '.join(current))
            current = [w]
        else:
            current.append(w)
    if current:
        lines_out.append(' '.join(current))
    return '\n'.join(lines_out)


def build_urban_site(synth: dict) -> tuple:
    """Convert a synthetic site dict → (site_model, intersections).

    Runs the synthetic boundary + road list through the Phase 2 pipeline
    so full_urban_analysis sees the canonical site_model['roads'] structure.
    """
    sm = build_site_model(synth['site_boundary'])
    roads_result = analyze_roads(sm, synth['roads'])
    sm['roads'] = roads_result
    return sm, synth.get('intersections', [])


print('Imports OK')

Imports OK


C:\Users\tuemi\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
# ── Load sites ─────────────────────────────────────────────────────────────
# Try OSM fetch for Barcelona (INTERESTING_SITES[0]); fall back to synthetic.

bcn_preset = INTERESTING_SITES[0]  # Eixample, Barcelona
print(f'Attempting OSM fetch: {bcn_preset["name"]} '
      f'(lat={bcn_preset["lat"]}, lon={bcn_preset["lon"]})...')
osm_site = fetch_or_fallback(
    bcn_preset['lat'], bcn_preset['lon'],
    radius_m=bcn_preset['radius_m'],
    timeout=10,
    fallback_index=0,
)
print(f'  source={osm_site["source"]} | roads={osm_site["road_count"]} | '
      f'intersections={osm_site["intersection_count"]}')

# Three synthetic demo sites (each exercises a different site type)
synth_a = SYNTHETIC_SITES[0]  # crossroads_corner
synth_b = SYNTHETIC_SITES[1]  # t_junction_terminal
synth_c = SYNTHETIC_SITES[2]  # triangular_corner

sm_a, ixs_a = build_urban_site(synth_a)
sm_b, ixs_b = build_urban_site(synth_b)
sm_c, ixs_c = build_urban_site(synth_c)

result_a = full_urban_analysis(sm_a, intersections=ixs_a)
result_b = full_urban_analysis(sm_b, intersections=ixs_b)
result_c = full_urban_analysis(sm_c, intersections=ixs_c)

SITES = [
    ('A', synth_a, sm_a, result_a, 'Crossroads Corner (Eixample-style)'),
    ('B', synth_b, sm_b, result_b, 'T-Junction Terminal (Bloomsbury-style)'),
    ('C', synth_c, sm_c, result_c, 'Triangular Corner (Flatiron-style)'),
]

print()
for tag, synth, sm, res, label in SITES:
    print(f'Site {tag}: {label}')
    print(f'  type={res["site_type"]} | frontages={res["frontage_count"]} | '
          f'corners={len(res["corner_conditions"])} | junctions={len(res["nearby_intersections"])}')
    for f in res['frontages']:
        print(f'    side {f["side_index"]}: {f["road_hierarchy"]:10s} '
              f'"{f["road_name"]}"  vis={f["visibility_score"]:.2f}  '
              f'access={f["recommended_access"]}')

Attempting OSM fetch: Eixample chamfered crossroads, Barcelona (lat=41.3936, lon=2.1628)...


  source=synthetic | roads=3 | intersections=1

Site A: Crossroads Corner (Eixample-style)
  type=crossroads_corner | frontages=2 | corners=1 | junctions=1
    side 0: main       "Carrer Gran"  vis=0.91  access=pedestrian
    side 4: secondary  "Avinguda Nord"  vis=0.58  access=primary_vehicle
Site B: T-Junction Terminal (Bloomsbury-style)
  type=t_junction_terminal | frontages=1 | corners=0 | junctions=1
    side 0: secondary  "High Street"  vis=0.69  access=primary_vehicle
Site C: Triangular Corner (Flatiron-style)
  type=triangular_corner | frontages=2 | corners=1 | junctions=1
    side 0: main       "Broadway"  vis=0.91  access=pedestrian
    side 1: main       "5th Avenue"  vis=0.88  access=pedestrian


C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio\team_04\agent\tools\osm_context.py:482: UserWarning: OSM fetch failed (Overpass API request failed: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter); using synthetic fallback.
  warnings.warn(f"OSM fetch failed ({exc}); using synthetic fallback.")


In [4]:
# ── §1  Road network ───────────────────────────────────────────────────────
# Three sites side-by-side; roads coloured by hierarchy (looks like a real city).

def draw_road_network(ax, synth: dict, sm: dict, result: dict, title: str):
    boundary = [(p[0], p[1]) for p in sm['boundary']]
    xs, ys   = zip(*boundary)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    ax.fill(xs, ys, alpha=0.15, color='#457B9D', zorder=1)

    for road in synth['roads']:
        cl  = road['centerline']
        clr = HIER_COLOUR.get(road['hierarchy'], '#888')
        rxs = [p[0] for p in cl]
        rys = [p[1] for p in cl]
        ls  = (0, (4, 2)) if road['hierarchy'] == 'path' else '-'
        # Width buffer
        ax.plot(rxs, rys, '-', color=clr, lw=road['width_m'] * 0.30,
                alpha=0.18, solid_capstyle='round', zorder=2)
        # Centreline
        ax.plot(rxs, rys, ls=ls, color=clr, lw=max(road['width_m'] * 0.10, 0.8),
                alpha=0.9, solid_capstyle='round', zorder=3)
        # Road name
        mx = (cl[0][0] + cl[-1][0]) / 2
        my = (cl[0][1] + cl[-1][1]) / 2
        ax.annotate(road['name'], xy=(mx, my), fontsize=5.5, ha='center',
                    color=clr, zorder=5,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.6, ec='none'))

    ax.plot(list(xs) + [xs[0]], list(ys) + [ys[0]], '-', color='#1D3557', lw=1.5, zorder=6)
    cx, cy = sum(xs) / len(xs), sum(ys) / len(ys)
    ax.text(cx, cy, 'SITE', ha='center', va='center', fontsize=7,
            color='#1D3557', fontweight='bold', zorder=7)

    for ix in result['nearby_intersections']:
        ax.plot(ix['point'][0], ix['point'][1],
                marker=IX_MARKER.get(ix['type'], 'o'),
                color=IX_COLOUR.get(ix['type'], '#888'),
                ms=8, zorder=8, mec='white', mew=0.8)

    ax.set_xlabel('m', fontsize=7); ax.set_ylabel('m', fontsize=7)
    ax.tick_params(labelsize=7)


fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (tag, synth, sm, result, label) in zip(axes, SITES):
    draw_road_network(ax, synth, sm, result, f'Site {tag}  {label}')

fig.legend(handles=[
    Line2D([0],[0], color=HIER_COLOUR['main'],      lw=3,   label='Main road'),
    Line2D([0],[0], color=HIER_COLOUR['secondary'], lw=2,   label='Secondary road'),
    Line2D([0],[0], color=HIER_COLOUR['path'],      lw=1.5, ls='--', label='Path'),
    Line2D([0],[0], color=IX_COLOUR['crossroads'],  marker='X', ms=8, ls='', label='Crossroads'),
    Line2D([0],[0], color=IX_COLOUR['t_junction'],  marker='v', ms=8, ls='', label='T-junction'),
], loc='lower center', ncol=5, fontsize=8, framealpha=0.9)
plt.suptitle('§1  Road Network — three distinct urban conditions',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.07, 1, 0.97])
plt.savefig('urban_context_1_road_network.png', dpi=120)
plt.show()
print('§1 complete')

§1 complete


C:\Users\tuemi\AppData\Local\Temp\ipykernel_30576\4217665891.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── §2  Frontage analysis ──────────────────────────────────────────────────
# Site sides coloured by visibility score; road centrelines offset outward.

def draw_frontage_analysis(ax, synth: dict, sm: dict, result: dict, title: str):
    boundary     = sm['boundary']
    corners_data = sm.get('corners', [])
    sides_data   = sm.get('sides', [])
    frontages    = {f['side_index']: f for f in result['frontages']}

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    bxs = [p[0] for p in boundary]
    bys = [p[1] for p in boundary]
    ax.fill(bxs, bys, alpha=0.08, color='#457B9D', zorder=1)

    for side in sides_data:
        si = side['edge_index']
        p1 = corners_data[side['from_node_index']]['point']
        p2 = corners_data[side['to_node_index']]['point']
        f  = frontages.get(si)
        if f:
            # Outward normal for pseudo-road centreline
            dx, dy = p2[0] - p1[0], p2[1] - p1[1]
            L = (dx**2 + dy**2) ** 0.5 or 1.0
            ox, oy = -dy / L * 14, dx / L * 14
            hier_clr = HIER_COLOUR.get(f['road_hierarchy'], '#888')
            rlw = 4 if f['road_hierarchy'] == 'main' else (2.5 if f['road_hierarchy'] == 'secondary' else 1.5)
            ax.plot([p1[0]+ox, p2[0]+ox], [p1[1]+oy, p2[1]+oy],
                    '-', color=hier_clr, lw=rlw * 2.5, alpha=0.12, solid_capstyle='round', zorder=2)
            ax.plot([p1[0]+ox, p2[0]+ox], [p1[1]+oy, p2[1]+oy],
                    '-', color=hier_clr, lw=rlw, alpha=0.9, solid_capstyle='round', zorder=3)
            # Frontage side (visibility heatmap)
            clr = _vis_colour(f['visibility_score'])
            lw  = 4 + f['visibility_score'] * 5
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                    '-', color=clr, lw=lw, alpha=0.85, solid_capstyle='round', zorder=4)
            mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
            ax.annotate(f'{f["visibility_score"]:.0%}', xy=(mx, my),
                        fontsize=7, ha='center', va='center', fontweight='bold',
                        color='#1D3557', zorder=6,
                        bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.75, ec='none'))
        else:
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                    '-', color='#CCCCCC', lw=1.5, alpha=0.5, zorder=4)

    ax.plot(bxs, bys, '--', color='#1D3557', lw=1, alpha=0.5, zorder=5)
    cx = sum(bxs[:-1]) / max(len(bxs) - 1, 1)
    cy = sum(bys[:-1]) / max(len(bys) - 1, 1)
    st_label = result['urban_response'].get('site_type_label', result['site_type'])
    ax.text(cx, cy, st_label, ha='center', va='center', fontsize=7, fontweight='bold',
            color='#1D3557', zorder=7,
            bbox=dict(boxstyle='round,pad=0.3', fc='#E9F1F7', alpha=0.85, ec='#1D3557', lw=0.8))
    ax.set_xlabel('m', fontsize=7); ax.set_ylabel('m', fontsize=7)
    ax.tick_params(labelsize=7)


fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (tag, synth, sm, result, label) in zip(axes, SITES):
    draw_frontage_analysis(ax, synth, sm, result, f'Site {tag}  {label}')

sm_img = plt.cm.ScalarMappable(cmap=VIS_CMAP, norm=plt.Normalize(0, 1))
sm_img.set_array([])
cbar = fig.colorbar(sm_img, ax=axes, orientation='horizontal',
                    pad=0.06, fraction=0.025, aspect=50)
cbar.set_label('Frontage visibility score  (0 = low public presence → 1 = high)', fontsize=8)

plt.suptitle('§2  Frontage Analysis — site sides coloured by visibility score',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.10, 1, 0.97])
plt.savefig('urban_context_2_frontage.png', dpi=120)
plt.show()
print('§2 complete')

C:\Users\tuemi\AppData\Local\Temp\ipykernel_30576\1642214628.py:69: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.10, 1, 0.97])


§2 complete


C:\Users\tuemi\AppData\Local\Temp\ipykernel_30576\1642214628.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── §3  Intersection classification & corner conditions ────────────────────

def draw_intersections_and_corners(ax, synth: dict, sm: dict, result: dict, title: str):
    boundary = sm['boundary']
    bxs = [p[0] for p in boundary]
    bys = [p[1] for p in boundary]

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    ax.fill(bxs, bys, alpha=0.10, color='#457B9D', zorder=1)
    ax.plot(bxs, bys, '-', color='#1D3557', lw=1.2, alpha=0.6, zorder=2)

    for road in synth['roads']:
        cl  = road['centerline']
        clr = HIER_COLOUR.get(road['hierarchy'], '#888')
        ax.plot([p[0] for p in cl], [p[1] for p in cl],
                '-', color=clr, lw=1.5, alpha=0.35, zorder=2)

    for ix in result['nearby_intersections']:
        ix_col = IX_COLOUR.get(ix['type'], '#888')
        ms = 8 + ix['degree'] * 2
        ax.plot(ix['point'][0], ix['point'][1],
                marker=IX_MARKER.get(ix['type'], 'o'),
                color=ix_col, ms=ms, zorder=8, mec='white', mew=1, alpha=0.9)
        ax.annotate(
            f"{ix['type'].replace('_',' ')}\ndeg={ix['degree']}, {ix['distance_to_site_m']:.0f} m",
            xy=(ix['point'][0], ix['point'][1]),
            xytext=(8, 8), textcoords='offset points',
            fontsize=6, color=ix_col, zorder=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75, ec='none'),
        )

    for cc in result['corner_conditions']:
        vis  = cc['visibility_score']
        r    = 7 + vis * 8
        fill = _vis_colour(vis)
        edge = '#E63946' if cc['is_gateway'] else '#2A9D8F'
        lw   = 2.5 if cc['is_gateway'] else 1.0
        ax.plot(cc['point'][0], cc['point'][1], 'o',
                color=fill, ms=r, mec=edge, mew=lw, zorder=10)
        lbl = 'G' if cc['is_gateway'] else f'{vis:.0%}'
        ax.text(cc['point'][0], cc['point'][1], lbl,
                ha='center', va='center', fontsize=6, fontweight='bold',
                color='#1D3557', zorder=11)

    ax.set_xlabel('m', fontsize=7); ax.set_ylabel('m', fontsize=7)
    ax.tick_params(labelsize=7)


fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (tag, synth, sm, result, label) in zip(axes, SITES):
    draw_intersections_and_corners(ax, synth, sm, result, f'Site {tag}  {label}')

fig.legend(handles=[
    Line2D([0],[0], marker='X', color=IX_COLOUR['crossroads'], ms=10, ls='', label='Crossroads'),
    Line2D([0],[0], marker='v', color=IX_COLOUR['t_junction'], ms=10, ls='', label='T-junction'),
    Line2D([0],[0], marker='o', color=(0.8, 0.7, 0.3), ms=10, ls='', mec='#E63946', mew=2, label='Gateway corner'),
    Line2D([0],[0], marker='o', color=(0.8, 0.7, 0.3), ms=10, ls='', mec='#2A9D8F', mew=1, label='Corner (non-gateway)'),
], loc='lower center', ncol=4, fontsize=8, framealpha=0.9)
plt.suptitle('§3  Intersection Classification & Corner Conditions',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.07, 1, 0.97])
plt.savefig('urban_context_3_intersections.png', dpi=120)
plt.show()

print('§3 complete')
for tag, synth, sm, result, label in SITES:
    ccs = result['corner_conditions']
    gw  = sum(1 for c in ccs if c['is_gateway'])
    print(f'  Site {tag}: {len(ccs)} corners ({gw} gateway)')

§3 complete
  Site A: 1 corners (1 gateway)
  Site B: 0 corners (0 gateway)
  Site C: 1 corners (1 gateway)


C:\Users\tuemi\AppData\Local\Temp\ipykernel_30576\2489745942.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── §4  Access point recommendations ──────────────────────────────────────

def draw_access(ax, synth: dict, sm: dict, result: dict, title: str):
    boundary     = sm['boundary']
    corners_data = sm.get('corners', [])
    sides_data   = sm.get('sides', [])
    frontages    = {f['side_index']: f for f in result['frontages']}
    bxs = [p[0] for p in boundary]
    bys = [p[1] for p in boundary]

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)
    ax.fill(bxs, bys, alpha=0.10, color='#457B9D', zorder=1)
    ax.plot(bxs, bys, '-', color='#1D3557', lw=1.5, zorder=2)

    for road in synth['roads']:
        cl  = road['centerline']
        clr = HIER_COLOUR.get(road['hierarchy'], '#888')
        ax.plot([p[0] for p in cl], [p[1] for p in cl],
                '-', color=clr, lw=1.5, alpha=0.35, zorder=2)

    # Frontage sides (faint highlight)
    for side in sides_data:
        si = side['edge_index']
        if si not in frontages:
            continue
        p1 = corners_data[side['from_node_index']]['point']
        p2 = corners_data[side['to_node_index']]['point']
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                '-', color=HIER_COLOUR.get(frontages[si]['road_hierarchy'], '#888'),
                lw=4, alpha=0.25, zorder=3, solid_capstyle='round')

    SYMBOLS = {'vehicle': '▶', 'pedestrian': '♟', 'service': '⚙'}
    access  = result['access']
    for cat in ('vehicle', 'pedestrian', 'service'):
        for ap in access.get(cat, []):
            pt  = ap['point']
            col = ACCESS_COLOUR[cat]
            ax.plot(pt[0], pt[1], 'o', color=col, ms=12, zorder=8, alpha=0.9)
            ax.text(pt[0], pt[1], SYMBOLS[cat],
                    ha='center', va='center', fontsize=8, zorder=9, color='white')
            ax.annotate(f'{cat[:3]}: {ap["notes"][:30]}',
                        xy=(pt[0], pt[1]), xytext=(10, -12),
                        textcoords='offset points', fontsize=5.5, color=col, zorder=10,
                        bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.7, ec='none'))

    ax.set_xlabel('m', fontsize=7); ax.set_ylabel('m', fontsize=7)
    ax.tick_params(labelsize=7)


fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, (tag, synth, sm, result, label) in zip(axes, SITES):
    draw_access(ax, synth, sm, result, f'Site {tag}  {label}')

fig.legend(handles=[
    Line2D([0],[0], marker='o', color=ACCESS_COLOUR['vehicle'],    ms=10, ls='', label='Vehicle entry'),
    Line2D([0],[0], marker='o', color=ACCESS_COLOUR['pedestrian'], ms=10, ls='', label='Pedestrian entry'),
    Line2D([0],[0], marker='o', color=ACCESS_COLOUR['service'],    ms=10, ls='', label='Service entry'),
], loc='lower center', ncol=3, fontsize=9, framealpha=0.9)
plt.suptitle('§4  Access Recommendations — vehicle / pedestrian / service',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.07, 1, 0.97])
plt.savefig('urban_context_4_access.png', dpi=120)
plt.show()

print('§4 complete')
for tag, synth, sm, result, label in SITES:
    acc = result['access']
    print(f'  Site {tag}: {acc["vehicle_count"]}V  '
          f'{acc["pedestrian_count"]}P  {acc["service_count"]}S')

§4 complete
  Site A: 1V  2P  0S
  Site B: 1V  1P  0S
  Site C: 0V  2P  1S


C:\Users\tuemi\AppData\Local\Temp\ipykernel_30576\697576189.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── §5  Urban response text panels ────────────────────────────────────────
# Note: _wrap() is defined in the imports cell (cell-2)

PANEL_BG = {'A': '#FFF3E0', 'B': '#E8F5E9', 'C': '#EDE7F6'}
PANEL_HD = {'A': '#E65100', 'B': '#1B5E20', 'C': '#4527A0'}

fig, axes = plt.subplots(1, 3, figsize=(16, 7))

for ax, (tag, synth, sm, result, label) in zip(axes, SITES):
    resp = result['urban_response']
    acc  = result['access']
    ax.set_facecolor(PANEL_BG[tag])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    def _t(x, y, txt, fs=8, fw='normal', clr='#333333'):
        ax.text(x, y, txt, transform=ax.transAxes, fontsize=fs,
                fontweight=fw, color=clr, va='top', multialignment='left')

    _t(0.05, 0.94, f'Site {tag}', 13, 'bold', PANEL_HD[tag])
    _t(0.05, 0.88, resp.get('site_type_label', ''), 10, 'bold')
    _t(0.05, 0.82, '─' * 50, 8, 'normal', '#AAAAAA')
    _t(0.05, 0.77, 'Building response:', 8, 'bold', '#555')
    _t(0.07, 0.72, _wrap(resp.get('building_response', ''), 58))
    _t(0.05, 0.62, 'Massing strategy:', 8, 'bold', '#555')
    _t(0.07, 0.57, _wrap(resp.get('massing_strategy', ''), 58))
    _t(0.05, 0.47, 'Entry strategy:', 8, 'bold', '#555')
    _t(0.07, 0.42, _wrap(resp.get('entry_strategy', ''), 58))
    _t(0.05, 0.32, 'Facade strategy:', 8, 'bold', '#555')
    _t(0.07, 0.27, _wrap(resp.get('facade_strategy', ''), 58))
    if resp.get('corner_treatment'):
        _t(0.05, 0.17, 'Corner treatment:', 8, 'bold', '#555')
        _t(0.07, 0.12, _wrap(resp['corner_treatment'], 58))

    strip = (f"{result['frontage_count']} frontage(s)  ·  "
             f"{acc['vehicle_count']}V {acc['pedestrian_count']}P {acc['service_count']}S access")
    ax.text(0.5, 0.03, strip, transform=ax.transAxes, fontsize=7,
            ha='center', color=PANEL_HD[tag], fontweight='bold')

plt.suptitle('§5  Urban Response — architectural strategy per site type',
             fontsize=12, fontweight='bold', y=0.99)
plt.tight_layout()
plt.savefig('urban_context_5_response.png', dpi=120)
plt.show()
print('§5 complete')

§5 complete


C:\Users\tuemi\AppData\Local\Temp\ipykernel_30576\1527415117.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ── §6  Geometric intersection detection ───────────────────────────────────
# detect_intersections_from_roads() recovers junction types purely from road
# centreline geometry — no pre-labelled OSM nodes required.

print('=== Geometric intersection detection (offline) ===')
print()
for tag, synth, sm, result, label in SITES:
    detected = detect_intersections_from_roads(synth['roads'])
    print(f'Site {tag}: {label}')
    if not detected:
        print('  (no intersections within road extents)')
    for ix in detected:
        print(f'  geometric: {ix["type"]:22s} deg={ix["degree"]}  '
              f'at ({ix["point"][0]:.1f}, {ix["point"][1]:.1f})')
    pre = synth.get('intersections', [])
    print(f'  pre-baked: ', end='')
    for ix in pre:
        print(f'{ix["type"]} at ({ix["point"][0]}, {ix["point"][1]})', end='  ')
    print()
    print()

=== Geometric intersection detection (offline) ===

Site A: Crossroads Corner (Eixample-style)
  geometric: crossroads             deg=4  at (-12.0, -12.0)
  geometric: crossroads             deg=4  at (-12.0, 55.0)
  pre-baked: crossroads at (-12, -12)  

Site B: T-Junction Terminal (Bloomsbury-style)
  geometric: t_junction             deg=3  at (0.0, 0.0)
  geometric: crossroads             deg=4  at (-50.0, 0.0)
  geometric: crossroads             deg=4  at (-50.0, 73.0)
  pre-baked: t_junction at (0, 0)  

Site C: Triangular Corner (Flatiron-style)
  (no intersections within road extents)
  pre-baked: crossroads at (57, -12)  



In [10]:
# ── §7  Summary table ──────────────────────────────────────────────────────

W = 92
print('=' * W)
print(f'{"":3} {"Site type":30} {"Frontages":>10} {"Corners":>9} {"Junctions":>10} {"V/P/S":>8}')
print('-' * W)
for tag, synth, sm, result, label in SITES:
    acc  = result['access']
    ccs  = result['corner_conditions']
    gws  = sum(1 for c in ccs if c['is_gateway'])
    ixn  = len(result['nearby_intersections'])
    typ  = result['urban_response']['site_type_label']
    print(f'{tag:<3} {typ:30} '
          f'{result["frontage_count"]:>10} '
          f'{len(ccs):>5} ({gws}G) '
          f'{ixn:>10} '
          f'{acc["vehicle_count"]:>3}/{acc["pedestrian_count"]}/{acc["service_count"]}')
print('=' * W)
print()
print('Phase 2b COMPLETE.')
print('  All results are offline-safe (synthetic sites, no network required).')
print('  OSM fetch via fetch_or_fallback() adds real city data when online.')
print()
print('Integration:')
print('  Phase 2  → road_context.analyze_roads() tags site sides with adjacent roads')
print('  Phase 2b → full_urban_analysis() classifies site type + generates design response')
print('  Phase 3  → derive_site_grid() uses main_road_side_index for grid alignment')
print('  Frontend → UrbanAnalysisOverlay.tsx renders all layers as SVG')

    Site type                       Frontages   Corners  Junctions    V/P/S
--------------------------------------------------------------------------------------------
A   Crossroads Corner                       2     1 (1G)          1   1/2/0
B   T-Junction Terminal (Focal Terminus)          1     0 (0G)          1   1/1/0
C   Triangular Corner                       2     1 (1G)          1   0/2/1

Phase 2b COMPLETE.
  All results are offline-safe (synthetic sites, no network required).
  OSM fetch via fetch_or_fallback() adds real city data when online.

Integration:
  Phase 2  → road_context.analyze_roads() tags site sides with adjacent roads
  Phase 2b → full_urban_analysis() classifies site type + generates design response
  Phase 3  → derive_site_grid() uses main_road_side_index for grid alignment
  Frontend → UrbanAnalysisOverlay.tsx renders all layers as SVG
